In [ ]:
# Contributors: [Group 5 — fill in names]
# Course: Recommendation Engines, IE University 2025-26
# Notebook 03: Content-Based Recommender

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
from utils import *

In [ ]:
import gc
import ast
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler, MultiLabelBinarizer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine
from sklearn.manifold import TSNE
from scipy.sparse import csr_matrix, hstack, issparse

np.random.seed(42)

## 1. Data Loading & Train/Test Split

In [ ]:
df, df_meta = load_data()
print(f'Reviews: {len(df):,} | Products: {len(df_meta):,}')

In [ ]:
train_df, test_df, split_info = temporal_train_test_split(df)
train_items = split_info['train_items']
cold_items  = split_info['cold_items']

In [ ]:
lookups = build_lookup_structures(train_df, test_df)
GLOBAL_MEAN      = lookups['global_mean']
train_user_items = lookups['train_user_items']

sample_users = sample_eval_users(lookups, n=1000)

## 2. Feature Engineering

For each item we build a hybrid feature vector:
- **TF-IDF** on title + description (5,000 dimensions, `sublinear_tf=True`)
- **Categories** — multi-hot encoding of leaf categories (top 500)
- **Brand** — one-hot encoding (top 100 brands + "Other")
- **Price** — MinMax-normalised to [0, 1]

The resulting sparse matrix has shape `(n_train_items, ~5,601)`.

In [ ]:
# Align metadata with training items
meta = df_meta.rename(columns={'asin': 'item_id'}).drop_duplicates(subset='item_id')
all_train_items_list = list(train_items)
meta = meta.set_index('item_id').reindex(all_train_items_list).reset_index()
meta['title']       = meta['title'].fillna('')
meta['description'] = meta['description'].fillna('')
meta['text']        = meta['title'] + ' ' + meta['description']

print(f'Items with text: {(meta["text"].str.len() > 0).sum():,} / {len(meta):,}')

In [ ]:
print('Building TF-IDF matrix (5000 features)...')
tfidf = TfidfVectorizer(max_features=5000, stop_words='english', sublinear_tf=True, dtype=np.float32)
tfidf_matrix = tfidf.fit_transform(meta['text'])
print(f'TF-IDF: {tfidf_matrix.shape}, nnz={tfidf_matrix.nnz:,}')

In [ ]:
def parse_categories(cat_str):
    try:
        cats = ast.literal_eval(str(cat_str))
        return [path[-1] for path in cats if isinstance(path, list) and path]
    except Exception:
        return []

meta['parsed_cats'] = meta['categories'].apply(parse_categories)

# Top-500 leaf categories (multi-hot)
all_cats  = [c for cats in meta['parsed_cats'] for c in cats]
top_cats  = [c for c, _ in Counter(all_cats).most_common(500)]
top_cats_set = set(top_cats)
meta['filtered_cats'] = meta['parsed_cats'].apply(lambda x: [c for c in x if c in top_cats_set])
mlb = MultiLabelBinarizer(classes=top_cats, sparse_output=True)
cat_matrix = mlb.fit_transform(meta['filtered_cats'])
print(f'Category matrix: {cat_matrix.shape}')

In [ ]:
# Brand: top-100 + 'Other'
meta['brand'] = meta['brand'].fillna('Unknown')
top_brands = meta['brand'].value_counts().head(100).index.tolist()
meta['brand_clean'] = meta['brand'].apply(lambda x: x if x in top_brands else 'Other')
brand_matrix = csr_matrix(pd.get_dummies(meta['brand_clean'], sparse=True, dtype=np.float32).values)
print(f'Brand matrix: {brand_matrix.shape}')

# Price: normalised
meta['price'] = pd.to_numeric(meta['price'], errors='coerce').fillna(meta['price'].median())
price_matrix  = csr_matrix(MinMaxScaler().fit_transform(meta[['price']].values), dtype=np.float32)

# Combined feature matrix
item_features = hstack([tfidf_matrix, cat_matrix, brand_matrix, price_matrix]).tocsr()
print(f'\nCombined item features: {item_features.shape}, nnz={item_features.nnz:,}')

item_to_idx = {item: i for i, item in enumerate(all_train_items_list)}
idx_to_item = {i: item for item, i in item_to_idx.items()}
gc.collect()

## 3. Content-Based Prediction Model

**Prediction**: cosine-similarity-weighted average of the ratings the user gave to the K most
content-similar items they have already interacted with.

**Recommendation**: build a user profile vector (mean of rated item features, weighted by
rating residual), then retrieve nearest neighbours in the feature space via KNN.

In [ ]:
# Damped popularity fallback for cold-start users
item_stats = train_df.groupby('item_id')['rating'].agg(['count', 'mean'])
item_stats['damped_score'] = (
    (item_stats['count'] * item_stats['mean'] + DAMPING_FACTOR * GLOBAL_MEAN)
    / (item_stats['count'] + DAMPING_FACTOR)
)
damped_means_series = item_stats['damped_score']
top_items_damped    = list(damped_means_series.sort_values(ascending=False).head(200).index)

print('Fitting NearestNeighbors on item features (n_neighbors=50)...')
nn_model = NearestNeighbors(n_neighbors=50, metric='cosine', algorithm='brute', n_jobs=-1)
nn_model.fit(item_features)
print('Done.')

In [ ]:
# Pre-build user→[(item_idx, rating)] map for fast lookup during prediction
train_user_data = {}
for uid, group in train_df.groupby('user_id'):
    pairs = [(item_to_idx[r.item_id], r.rating)
             for r in group.itertuples() if r.item_id in item_to_idx]
    if pairs:
        train_user_data[uid] = pairs
print(f'User profiles built: {len(train_user_data):,}')

In [ ]:
def predict_content_based(user_id, item_id, k=20):
    if item_id not in item_to_idx:
        return GLOBAL_MEAN
    if user_id not in train_user_data:
        return float(item_stats.loc[item_id, 'damped_score']) if item_id in item_stats.index else GLOBAL_MEAN

    target_idx  = item_to_idx[item_id]
    user_items  = train_user_data[user_id]
    user_idxs   = [idx for idx, _ in user_items]
    user_ratings = np.array([r for _, r in user_items])

    sims = sk_cosine(item_features[target_idx], item_features[user_idxs]).flatten()
    top_k = np.argsort(sims)[-k:] if len(sims) > k else np.arange(len(sims))
    top_sims, top_ratings = sims[top_k], user_ratings[top_k]
    mask = top_sims > 0
    if not mask.any():
        return GLOBAL_MEAN
    return float(np.clip(np.average(top_ratings[mask], weights=top_sims[mask]), 1.0, 5.0))


def get_cb_top_k(user_id, k=K):
    seen = train_user_items.get(user_id, set())
    if user_id not in train_user_data:
        return [it for it in top_items_damped if it not in seen][:k]

    user_items   = train_user_data[user_id]
    user_idxs    = [idx for idx, _ in user_items]
    weights      = np.array([r - GLOBAL_MEAN for _, r in user_items])
    user_profile = csr_matrix(weights.reshape(1, -1)) @ item_features[user_idxs]

    _, indices = nn_model.kneighbors(user_profile, n_neighbors=k + len(seen))
    recs = []
    for idx in indices.flatten():
        item_id = idx_to_item.get(idx)
        if item_id and item_id not in seen:
            recs.append(item_id)
        if len(recs) >= k:
            break
    return recs

## 4. Evaluation

Rating prediction uses a 20,000-row sample (the full 421K test set would take ~3 hours with
per-row cosine similarity lookups). Ranking metrics use 1,000 sampled users.

In [ ]:
TEST_SAMPLE_SIZE = 20000
print(f'Rating prediction on {TEST_SAMPLE_SIZE:,}-row sample...')
test_sample = test_df.sample(n=min(TEST_SAMPLE_SIZE, len(test_df)), random_state=42)

actual_cb    = test_sample['rating'].tolist()
predicted_cb = [
    predict_content_based(row.user_id, row.item_id)
    for row in test_sample.itertuples()
]

rmse_cb = rmse(actual_cb, predicted_cb)
mae_cb  = mae(actual_cb, predicted_cb)
print(f'  Content-Based — RMSE: {rmse_cb:.4f}, MAE: {mae_cb:.4f}')

In [ ]:
print('Computing ranking metrics (1000 users)...')
all_recs_cb, ranking_cb = evaluate_ranking(get_cb_top_k, sample_users, lookups)

print('Computing diversity...')
div_cb = diversity_intra_list(all_recs_cb, item_features, item_to_idx)

recs_cb_dict = {u: r for u, r in zip(sample_users, all_recs_cb)}
results_cb = {'RMSE': rmse_cb, 'MAE': mae_cb, 'Diversity': div_cb, **ranking_cb}

print_metrics('Content-Based', rmse_cb, mae_cb, {**ranking_cb, 'Diversity': div_cb})

## 5. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Actual vs predicted rating distribution
axes[0].hist(actual_cb,    bins=20, alpha=0.6, label='Actual',        color='steelblue', edgecolor='white')
axes[0].hist(predicted_cb, bins=20, alpha=0.6, label='Predicted (CB)', color='coral',   edgecolor='white')
axes[0].set_title('Content-Based: Actual vs Predicted Ratings')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')
axes[0].legend()

# Coverage vs diversity scatter (one point = one model for context)
axes[1].bar(['Content-Based'], [div_cb], color='seagreen', edgecolor='white')
axes[1].set_title('Intra-List Diversity (Content-Based)')
axes[1].set_ylabel('Diversity score (0–1)')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

### t-SNE of item embeddings

In [ ]:
sample_size = 5000
sample_idxs = np.random.choice(len(all_train_items_list), size=sample_size, replace=False)
sample_feats = item_features[sample_idxs].toarray()

sample_cats = []
for idx in sample_idxs:
    cats = meta.iloc[idx]['parsed_cats'] if idx < len(meta) else []
    sample_cats.append(cats[0] if cats else 'Unknown')

cat_counts = Counter(sample_cats)
top_8 = {c for c, _ in cat_counts.most_common(8)}
sample_colors = [c if c in top_8 else 'Other' for c in sample_cats]

print('Running t-SNE (perplexity=30, 1000 iterations)...')
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
emb  = tsne.fit_transform(sample_feats)

plt.figure(figsize=(11, 7))
unique_cats = sorted(set(sample_colors))
cmap = plt.cm.tab10(np.linspace(0, 1, len(unique_cats)))
for i, cat in enumerate(unique_cats):
    mask = [c == cat for c in sample_colors]
    plt.scatter(emb[mask, 0], emb[mask, 1], s=5, alpha=0.5, label=cat, color=cmap[i])
plt.legend(markerscale=4, fontsize=8)
plt.title('t-SNE of Item Content Features (5 K sample, coloured by leaf category)')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.tight_layout()
plt.show()
print('t-SNE complete.')

## 6. Save Results

In [ ]:
import os, pickle

output = {
    'results_cb':           results_cb,
    'item_features':        item_features,    # sparse matrix — used by notebook 05 for diversity/serendipity
    'item_to_idx':          item_to_idx,
    'idx_to_item':          idx_to_item,
    'all_train_items_list': all_train_items_list,
    'recs_cb':              recs_cb_dict,      # {user_id: [item_ids]}
    'sample_users':         sample_users,
}

os.makedirs('../results', exist_ok=True)
with open('../results/03_content_based.pkl', 'wb') as f:
    pickle.dump(output, f)

print('Saved: ../results/03_content_based.pkl')
print(f'item_features shape: {item_features.shape}')